# Framing hits — evidence & topics

Evidence phrases, TF-IDF, embeddings / clustering, topic merge, and CSV exports under **`03_Framing/outputs/`**.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
for _p in (_here, _here / "03_Framing", _here.parent / "03_Framing"):
    if (_p / "framing_paths.py").is_file():
        sys.path.insert(0, str(_p))
        break

import pandas as pd
from framing_paths import resolve_framing_paths, topic_table_candidates

THESIS_ROOT, FRAMING_ROOT, OUT_DIR = resolve_framing_paths()
FRAMING_CSV = OUT_DIR / "framing_gpt_results.csv"

df = pd.read_csv(FRAMING_CSV)
print(f"Loaded {len(df):,} rows from {FRAMING_CSV}")


In [ ]:
df.info()

In [ ]:
df = df[['hit_id', 'row_id', 'source', 'Title', 'hit_text', 'context_idx', 'context_window', 'category', 'evidence']]


In [ ]:
df= df[df['evidence'].notna()]

In [ ]:
df.info()

#### Split compound hit labels

Some `hit_text` values combine several labels with `|` (e.g. `Spiegel | Qualitätsmedien`, `ZDF | Jan Böhmermann`). Those are split on `|`, trimmed, and **exploded** so each label is counted separately in the frequency tables below.


In [ ]:
def split_hit_text_to_segments(s):
    """Split pipe-separated hit strings into individual labels."""
    if not isinstance(s, str) or not str(s).strip():
        return []
    return [p.strip() for p in str(s).split("|") if str(p).strip()]


df_for_hits = (
    df.assign(hit_segment=df["hit_text"].map(split_hit_text_to_segments))
    .explode("hit_segment", ignore_index=True)
)
df_for_hits = df_for_hits.dropna(subset=["hit_segment"])

# Example: original vs exploded row count
print(f"Rows before split/explode: {len(df):,}")
print(f"Rows after split/explode (one row per segment): {len(df_for_hits):,}")


#### Extra keyword hits per outlet

Count of **segment** hits (`hit_segment` in `df_for_hits`) whose text **exactly matches** one of
`EXTRA_KEYWORDS` (case-insensitive, after stripping whitespace).


In [ ]:
EXTRA_KEYWORDS = [
    "Mainstreammedien",
    "Staatsmedien",
    "Staatsfunk",
    "Qualitätsmedien",
    "Staatssender",
    "Lügenpresse",
    "Haltungsjournalisten",
    "Gleichschaltung",
    "Mainstreampresse",
    "Altmedien",
    "Systemmedien",
    "Qualitätsjournalismus",
    "Alternativmedien",
    "Gesternmedien",
    "Haltungsmedien",
    "Westmedien",
    "Regierungsmedien",
    "Linkspresse",
    "Qualitätspresse",
    "Haltungsjournalismus",
    "Propagandamedien",
    "Propagandasender",
    "Propagandamaschine",
    "Medienpropaganda",
    "Staatsrundfunk",
]

_kw_set = {w.casefold() for w in EXTRA_KEYWORDS}
_seg = df_for_hits["hit_segment"].astype(str).str.strip().str.casefold()
_is_kw = _seg.isin(_kw_set)

extra_keyword_hits_per_outlet = (
    df_for_hits.loc[_is_kw]
    .groupby("source", observed=True)
    .size()
    .reset_index(name="n_extra_keyword_hits")
)
segments_per_outlet = (
    df_for_hits.groupby("source", observed=True)
    .size()
    .reset_index(name="n_segments")
)
extra_keyword_summary = segments_per_outlet.merge(
    extra_keyword_hits_per_outlet, on="source", how="left"
)
extra_keyword_summary["n_extra_keyword_hits"] = (
    extra_keyword_summary["n_extra_keyword_hits"].fillna(0).astype(int)
)
extra_keyword_summary["share_of_segments_pct"] = (
    100.0 * extra_keyword_summary["n_extra_keyword_hits"] / extra_keyword_summary["n_segments"]
).round(2)
extra_keyword_summary = extra_keyword_summary.sort_values(
    "n_extra_keyword_hits", ascending=False
)

display(extra_keyword_summary)
print("Total extra-keyword segment hits (all outlets):", int(_is_kw.sum()))


### Top 10 hit labels per outlet (by frequency)

Uses **`df_for_hits`** (pipe-separated `hit_text` split into rows). Below: long table, then **wide table** — each outlet is a column group with **`word`** and **`count`** sub-columns (ranks 1–10), plus outlet totals.


In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd

hit_freq = (
    df_for_hits.groupby(["source", "hit_segment"], observed=True)
    .size()
    .reset_index(name="count")
)
top10_hits = (
    hit_freq.sort_values(["source", "count"], ascending=[True, False])
    .groupby("source", group_keys=False)
    .head(10)
    .reset_index(drop=True)
)
top10_hits["rank"] = top10_hits.groupby("source").cumcount() + 1
top10_hits = top10_hits.rename(columns={"hit_segment": "hit"})
top10_hits = top10_hits[["rank", "source", "hit", "count"]]

# Total rows per outlet after splitting (each segment counts as one)
counts_per_outlet = (
    df_for_hits.groupby("source", observed=True)
    .size()
    .reset_index(name="n_segments")
    .sort_values("n_segments", ascending=False)
)
display(counts_per_outlet)

fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(counts_per_outlet))))
ax.barh(counts_per_outlet["source"][::-1], counts_per_outlet["n_segments"][::-1], color="steelblue", edgecolor="black")
ax.set_xlabel("Number of hit segments (rows after split)")
ax.set_ylabel("Outlet")
ax.set_title("Hit segments per outlet (total)")
plt.tight_layout()
plt.show()

display(top10_hits)

# Wide format: one column group per outlet; each group has sub-columns (word, count)
outlet_order = counts_per_outlet["source"].tolist()
wide_parts = []
for src in outlet_order:
    sub = top10_hits.loc[top10_hits["source"] == src, ["rank", "hit", "count"]]
    if sub.empty:
        sub = pd.DataFrame(
            index=pd.RangeIndex(1, 11, name="rank"),
            columns=["hit", "count"],
        )
    else:
        sub = sub.set_index("rank")[["hit", "count"]]
    sub.columns = pd.MultiIndex.from_product([[src], ["word", "count"]])
    wide_parts.append(sub)

wide_top10 = pd.concat(wide_parts, axis=1).sort_index()
with pd.option_context("display.max_columns", None, "display.max_colwidth", 100, "display.width", 200):
    display(wide_top10)


In [ ]:
df_for_hits.info()

In [ ]:
# Top 10 hits overall (except Tagesschau) and their counts

# Exclude Tagesschau
top_hits_excl_tagesschau = df_for_hits[df_for_hits["source"] != "Tagesschau"]

# Group by hit_segment, sum counts, then get top 10
overall_top10 = (
    top_hits_excl_tagesschau
    .groupby("hit_segment", as_index=False)["count"]
    .sum()
    .sort_values("count", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

overall_top10 = overall_top10.rename(columns={"hit_segment": "hit"})

display(overall_top10)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Non-neutral framing only: exclude NEUTRAL and IRRELEVANT from numerator and denominator
framed = df[~df["category"].isin(["NEUTRAL", "IRRELEVANT"])].copy()
cat_counts = framed.groupby(["source", "category"]).size().unstack(fill_value=0)
cat_counts_pct = cat_counts.div(cat_counts.sum(axis=1), axis=0) * 100

# Rows = framing category, columns = outlet; values = share of that outlet's non-neutral hits
hm = cat_counts_pct.T
fig_h = max(6.0, 0.35 * len(hm.index))
fig, ax = plt.subplots(figsize=(12, fig_h))
sns.heatmap(
    hm,
    annot=True,
    fmt=".1f",
    cmap="YlOrRd",
    linewidths=0.3,
    ax=ax,
    cbar_kws={"label": "Share of outlet's non-neutral hits (%)"},
)
ax.set_xlabel("Outlet")
ax.set_ylabel("Framing category")
ax.set_title("Framing by outlet (share of non-neutral hits per outlet)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### Framing category × outlet × hit (segment counts)

Within each **framing category** (excluding `NEUTRAL` and `IRRELEVANT`), which **hit segments** (`hit_segment` after pipe-splitting) appear per **outlet**, and how often.

Uses the same **`df_for_hits`** as above (so it follows whatever row filter applies to `df` before the split cell). Heatmaps: rows = hit segments (top *N* by total count in that category), columns = outlets, cell value = count.


In [ ]:
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Non-neutral framing rows with exploded hit segments
framed_seg = df_for_hits[
    ~df_for_hits["category"].isin(["NEUTRAL", "IRRELEVANT"])
].copy()

hit_cat_counts = (
    framed_seg.groupby(["category", "source", "hit_segment"], observed=True)
    .size()
    .reset_index(name="n")
)

# Long-format table (sort by category, then count)
hit_cat_long = hit_cat_counts.sort_values(
    ["category", "n", "source"], ascending=[True, False, True]
)
with pd.option_context("display.max_rows", 80, "display.max_colwidth", 60):
    display(hit_cat_long)

# Heatmap per framing category (top hit segments in that category)
TOP_N_SEGMENTS = 10
MIN_CATEGORY_TOTAL = 3

cat_totals = hit_cat_counts.groupby("category")["n"].sum().sort_values(ascending=False)
cats_plot = cat_totals[cat_totals >= MIN_CATEGORY_TOTAL].index.tolist()

for cat in cats_plot:
    sub = hit_cat_counts[hit_cat_counts["category"] == cat]
    pivot = sub.pivot_table(
        index="hit_segment", columns="source", values="n", aggfunc="sum", fill_value=0
    )
    row_totals = pivot.sum(axis=1)
    top_idx = row_totals.nlargest(min(TOP_N_SEGMENTS, len(row_totals))).index
    pivot = pivot.reindex(top_idx)
    fig_h = max(5.0, 0.38 * len(pivot))
    fig_w = max(10.0, 0.9 * len(pivot.columns))
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    sns.heatmap(
        pivot,
        annot=True,
        fmt=".0f",
        cmap="YlOrRd",
        linewidths=0.3,
        ax=ax,
        cbar_kws={"label": "Count"},
    )
    ax.set_title(f"Hit segments by outlet — {cat}")
    ax.set_xlabel("Outlet")
    ax.set_ylabel("Hit (segment)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()



# Relation to Topics

In [ ]:
topic_path = None
for candidate in topic_table_candidates(THESIS_ROOT):
    if candidate.is_file():
        topic_path = candidate
        break
if topic_path is None:
    raise FileNotFoundError(
        "Could not find df_combined_topic_only_new.csv. "
        "Expected under 02_TopicModeling/outputs/ or 1a_BERTopic/outputs/."
    )
dft = pd.read_csv(topic_path)
print(f"Topic table: {topic_path} ({len(dft):,} rows)")


## Deep Dive into Evidences

In [ ]:
df_evidence = df.copy()


In [ ]:
import matplotlib.pyplot as plt

# Group by source and category, count mentions
cat_counts = df_evidence.groupby(['source', 'category']).size().unstack(fill_value=0)

# Remove NEUTRAL and IRRELEVANT if you want only accusations (as below)
cat_counts = cat_counts.drop(columns=[c for c in ['NEUTRAL', 'IRRELEVANT'] if c in cat_counts.columns], errors='ignore')

# Convert to percent per source
cat_counts_pct = (cat_counts.T / cat_counts.sum(axis=1)).T * 100

# Plot as stacked bar chart
fig, ax = plt.subplots(figsize=(12, 7))
cat_counts_pct.plot(kind='bar', stacked=True, ax=ax, colormap='tab20c', edgecolor='black')

# Add percentage labels on the bars
for idx, row in enumerate(cat_counts_pct.values):
    y_offset = 0
    for col_i, val in enumerate(row):
        if val > 2:  # Only show label if segment is not too small
            ax.text(
                idx, 
                y_offset + val/2, 
                f"{val:.1f}%", 
                ha='center', va='center', fontsize=8, color='white', fontweight='bold'
            )
        y_offset += val

ax.set_ylabel('Anteil pro Quelle (%)')
ax.set_xlabel('Quelle')
ax.set_title('Stacked Bar Chart: Evidenz-Kategorien je Outlet (prozentual)')
ax.legend(title='Kategorie', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

### TF-IDF on evidence phrases (one document per outlet)

Non-empty `evidence` strings are concatenated **per `source`**. Only rows with accusation/bias categories are kept (`NEUTRAL` and `IRRELEVANT` excluded). **Sublinear TF** reduces the advantage of very long outlet texts. Terms with **low IDF** recur across many outlets (shared accusation vocabulary); **high TF-IDF in one outlet** highlights terms concentrated there (distinctive versus the rest).


In [ ]:
from IPython.display import display

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# --- Build one pseudo-document per outlet (accusation-related evidence only) ---
work = df_evidence.copy()
work["evidence"] = work["evidence"].astype(str).str.strip()
work = work[work["evidence"] != ""]
work = work[~work["category"].isin(["NEUTRAL", "IRRELEVANT"])].copy()

docs_series = work.groupby("source", sort=True)["evidence"].agg(lambda parts: " ".join(parts))
sources = list(docs_series.index)
documents = [docs_series[s] for s in sources]

# German function words + short noise tokens (TF-IDF focuses on content-bearing terms)
GERMAN_STOP = {
    "der", "die", "das", "und", "oder", "ein", "eine", "einer", "einem", "einen",
    "ist", "sind", "war", "waren", "wird", "werden", "wurde", "wurden", "hat", "haben",
    "im", "in", "am", "an", "auf", "zu", "zum", "zur", "von", "vom", "mit", "nach",
    "als", "auch", "nicht", "nur", "wie", "so", "sich", "dass", "daß", "es", "er", "sie",
    "ihr", "ihre", "ihnen", "man", "bei", "aus", "über", "unter", "vor", "durch",
    "für", "gegen", "ohne", "um", "bis", "ob", "weil", "wenn", "noch", "schon",
    "mehr", "mich", "mir", "uns", "ihm", "ihn", "dem", "den", "des", "diese", "dieser",
    "dieses", "alle", "allem", "allen", "alles", "was", "wer", "wo", "wird", "sein",
    "seine", "seinem", "seinen", "seiner", "habe", "hast", "können", "kann",
    "müssen", "muß", "muss", "soll", "sollen", "wollen", "würde", "wäre", "hier", "dort",
    "ja", "nein", "all", "the", "pr",
    "ihren", "ihrer", "ihrem", "diesem", "diesen", "dessen", "deren",
}

vectorizer = TfidfVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b[\wäöüÄÖÜß]+\b",
    min_df=1,
    max_df=1.0,
    stop_words=list(GERMAN_STOP),
    sublinear_tf=True,
)
X = vectorizer.fit_transform(documents)
terms = np.array(vectorizer.get_feature_names_out())
idf = vectorizer.idf_

# --- Shared vocabulary: lowest IDF (appear in many / all outlet-documents) ---
n_shared = min(40, len(terms))
shared_order = np.argsort(idf)[:n_shared]
tfidf_shared = pd.DataFrame(
    {
        "term": terms[shared_order],
        "idf": idf[shared_order],
        "n_outlets_with_term": np.asarray(X[:, shared_order].getnnz(axis=0)).ravel(),
    }
)
display(tfidf_shared)

# --- Distinctive terms: highest TF-IDF per outlet ---
TOP_K = 25
rows = []
for i, src in enumerate(sources):
    col = X.getrow(i).toarray().ravel()
    order = np.argsort(-col)[:TOP_K]
    for rank, j in enumerate(order, start=1):
        if col[j] <= 0:
            continue
        rows.append(
            {
                "source": src,
                "rank": rank,
                "term": terms[j],
                "tfidf": col[j],
            }
        )

tfidf_by_outlet = pd.DataFrame(rows)
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
    "display.max_colwidth", None,
):
    display(tfidf_by_outlet)


### Evidence phrase embeddings + clustering

**Sentence model (same as BERTopic in this repo):** `paraphrase-multilingual-MiniLM-L12-v2` from [`bertopic_config.py`](../1a_BERTopic/bertopic_config.py) — multilingual MiniLM, strong German sentence similarity and used across your topic-modeling notebooks.

Non-empty `evidence` strings are embedded; vectors are **L2-normalized** and **K-means** is run with `n_clusters` in **8–12** (default **10**). Adjust `N_CLUSTERS` below if needed.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer

# Same default as thesis BERTopic pipeline (German + other languages)
EMBEDDING_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"
N_CLUSTERS = 6 

evid_work = df_evidence.copy()
evid_work["evidence"] = evid_work["evidence"].astype(str).str.strip()
evid_work = evid_work[evid_work["evidence"] != ""].reset_index(drop=True)
texts = evid_work["evidence"].tolist()
print(f"Embedding {len(texts):,} evidence phrases with {EMBEDDING_MODEL!r} …")

model = SentenceTransformer(EMBEDDING_MODEL)
embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=False,
)

X = normalize(embeddings, norm="l2", axis=1)
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
labels = kmeans.fit_predict(X)

evid_work["evidence_cluster"] = labels
df_evidence_clustered = evid_work  # use in later cells

sizes = (
    evid_work.groupby("evidence_cluster", sort=True)
    .size()
    .rename("n_phrases")
    .reset_index()
)
display(sizes)

# Representative phrases per cluster: highest cosine similarity to cluster centroid
centroids = kmeans.cluster_centers_
rows = []
for k in range(N_CLUSTERS):
    mask = labels == k
    idx = np.flatnonzero(mask)
    c = centroids[k]
    c = c / np.linalg.norm(c)
    sims = X[mask] @ c
    top_local = np.argsort(-sims)[:12]
    for rank, il in enumerate(top_local, start=1):
        gi = idx[il]
        rows.append(
            {
                "evidence_cluster": k,
                "rank_in_cluster": rank,
                "similarity_to_centroid": float(sims[il]),
                "evidence": texts[gi],
                "source": evid_work["source"].iloc[gi],
            }
        )

cluster_anchors = pd.DataFrame(rows)
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", None, "display.max_colwidth", 120):
    display(cluster_anchors)


In [ ]:
evid_work['evidence_cluster'].value_counts()

In [ ]:
evid_work.to_csv(OUT_DIR / "evidence_clusters_fullv2.csv", index=False)
print(f"Saved {OUT_DIR / 'evidence_clusters_fullv2.csv'}")


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import umap

# 2D map of normalized embeddings; colour = K-means cluster; stars = mean UMAP position per cluster
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=30,
    min_dist=0.08,
    metric="cosine",
    random_state=42,
)
X_2d = reducer.fit_transform(X)

centroids_2d = np.vstack(
    [X_2d[labels == k].mean(axis=0) for k in range(N_CLUSTERS)]
)

base = mpl.colormaps["tab20" if N_CLUSTERS > 10 else "tab10"]
palette = [base(i / max(N_CLUSTERS - 1, 1)) for i in range(N_CLUSTERS)]

fig, ax = plt.subplots(figsize=(11, 8))
for k in range(N_CLUSTERS):
    m = labels == k
    ax.scatter(
        X_2d[m, 0],
        X_2d[m, 1],
        c=[palette[k]],
        s=8,
        alpha=0.35,
        linewidths=0,
        label=f"cluster {k} (n={m.sum():,})",
        rasterized=True,
    )
ax.scatter(
    centroids_2d[:, 0],
    centroids_2d[:, 1],
    c=palette,
    s=240,
    marker="*",
    edgecolors="black",
    linewidths=0.6,
    zorder=5,
)
for k in range(N_CLUSTERS):
    ax.annotate(
        str(k),
        (centroids_2d[k, 0], centroids_2d[k, 1]),
        fontsize=9,
        fontweight="bold",
        ha="center",
        va="center",
        color="black",
        zorder=6,
    )
ax.set_title("Evidence phrases — UMAP of sentence embeddings (K-means clusters)")
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
ax.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=8,
    markerscale=0.8,
    frameon=True,
)
plt.tight_layout()
plt.show()

# cluster_anchors includes the outlet (source) for each anchor phrase
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", None,
    "display.max_colwidth", 100,
):
    display(cluster_anchors.sort_values(["evidence_cluster", "rank_in_cluster"]))


In [ ]:
# Save the cluster anchor table (including evidence_cluster and all assigned rows) as CSV

# Assumes `cluster_anchors` is a DataFrame containing all evidence phrases,
# with at least the columns ['evidence_cluster', ...].
# Save all rows, including the cluster assignments.

cluster_anchors.sort_values(["evidence_cluster", "rank_in_cluster"])
    .to_csv(OUT_DIR / "evidence_clusters_full.csv", index=False)
print(f"Saved {OUT_DIR / 'evidence_clusters_full.csv'} with all 10 clusters and assigned rows.")


### Outlet share per evidence cluster

100% stacked bars: within each cluster, the proportion of evidence phrases from each `source` (outlet).


In [ ]:
from IPython.display import display

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Requires `df_evidence_clustered` from the embedding cell (columns: evidence_cluster, source)
df = df_evidence_clustered.copy()
counts = (
    df.groupby(["evidence_cluster", "source"], observed=False)
    .size()
    .rename("n")
    .reset_index()
)
pivot = counts.pivot_table(
    index="evidence_cluster",
    columns="source",
    values="n",
    aggfunc="sum",
    fill_value=0,
)
pivot = pivot.sort_index()
outlets = sorted(pivot.columns)
pivot = pivot[outlets]
shares = pivot.div(pivot.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)

n_out = len(outlets)
base = mpl.colormaps["tab20" if n_out > 10 else "tab10"]
colors = [base(i / max(n_out - 1, 1)) for i in range(n_out)]

fig, ax = plt.subplots(figsize=(max(10, 0.9 * len(pivot.index)), 5.5))
x = np.arange(len(pivot.index))
bottom = np.zeros(len(pivot))
for i, outlet in enumerate(outlets):
    vals = shares[outlet].to_numpy()
    ax.bar(x, vals, bottom=bottom, label=outlet, color=colors[i], width=0.82)
    bottom += vals

ax.set_xticks(x)
ax.set_xticklabels([f"cluster {k}" for k in pivot.index])
ax.set_ylabel("share within cluster")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
ax.set_title("Outlet mix per evidence cluster (row-wise proportions)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

display(shares.map(lambda v: f"{v:.1%}"))


In [ ]:
# Compute bar chart of categories (BIAS) stacked per evidence_cluster

# Calculate counts of BIAS per evidence_cluster
bias_counts = (
    df.groupby(["evidence_cluster", "category"], observed=False)
    .size()
    .rename("n")
    .reset_index()
)
bias_pivot = bias_counts.pivot_table(
    index="evidence_cluster",
    columns="category",
    values="n",
    aggfunc="sum",
    fill_value=0,
)
bias_pivot = bias_pivot.sort_index()
categories = sorted(bias_pivot.columns)
bias_pivot = bias_pivot[categories]
bias_shares = bias_pivot.div(bias_pivot.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)

n_cat = len(categories)
base_cat = mpl.colormaps["tab20" if n_cat > 10 else "tab10"]
cat_colors = [base_cat(i / max(n_cat - 1, 1)) for i in range(n_cat)]

fig, ax = plt.subplots(figsize=(max(10, 0.9 * len(bias_pivot.index)), 5.5))
x = np.arange(len(bias_pivot.index))
bottom = np.zeros(len(bias_pivot))
for i, cat in enumerate(categories):
    vals = bias_shares[cat].to_numpy()
    ax.bar(x, vals, bottom=bottom, label=cat, color=cat_colors[i], width=0.82)
    bottom += vals

ax.set_xticks(x)
ax.set_xticklabels([f"cluster {k}" for k in bias_pivot.index])
ax.set_ylabel("share within cluster")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0))
ax.set_title("BIAS mix per evidence cluster (row-wise proportions)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

display(bias_shares.map(lambda v: f"{v:.1%}"))